In [1]:
%pip install streamlit

  Obtaining dependency information for streamlit from https://files.pythonhosted.org/packages/d7/8e/e635448a7fd6d92211d4ff2150356b8dffd3fcdc05c00511c61110078871/streamlit-1.64.0-py3-none-any.whl.metadata
  Obtaining dependency information for altair!=5.4.0,!=5.4.1,<7,>=5.0.0 from https://files.pythonhosted.org/packages/ca/b9/10a8bb13a0462e0bc4e02c0b7b8237dc24ab590f5bb4be94bec5e8f1d532/altair-6.3.0-py3-none-any.whl.metadata
  Obtaining dependency information for click<9,>=7.0 from https://files.pythonhosted.org/packages/58/50/6c0d534c5f134586a8e1ba4e330569e32f057e33372ae556463212fb4cd3/click-8.5.0-py3-none-any.whl.metadata
  Obtaining dependency information for pydeck<1,>=0.8.0b4 from https://files.pythonhosted.org/packages/6f/34/3998411437aff304a9ed4fa37a6fe1ef3132bcd2b5eac59851b80c86123c/pydeck-0.9.3-py2.py3-none-any.whl.metadata
  Obtaining dependency information for protobuf<8,>=5.26.1 from https://files.pythonhosted.org/packages/8a/55/b77bda4e5e5f5971fb51b07663694690e9afdb9402136c1

In [2]:
%%writefile app.py
from collections import defaultdict

import numpy as np
import pandas as pd
import statsmodels.api as sm
import streamlit as st
from scipy.stats import poisson

st.set_page_config(
    page_title="Premier League Score Predictor",
    page_icon="⚽",
    layout="centered",
)

st.title("⚽ Premier League Score Predictor")
st.caption("Poisson + Elo model using recent form and completed Premier League results.")

# Load the final trained models
home_model = sm.load("models/poisson_elo_home_goals.pickle")
away_model = sm.load("models/poisson_elo_away_goals.pickle")

# Load completed match history
matches = pd.read_csv(
    "data/processed/epl_matches_clean.csv",
    parse_dates=["Date"]
).sort_values("Date")

feature_columns = [
    "home_form_points_5",
    "away_form_points_5",
    "home_avg_goals_for_5",
    "away_avg_goals_for_5",
    "home_avg_goals_against_5",
    "away_avg_goals_against_5",
    "elo_difference",
]

team_history = defaultdict(list)
ratings = defaultdict(lambda: 1500.0)

K_FACTOR = 20
HOME_ADVANTAGE = 60

# Rebuild each team's current form and Elo rating
for _, match in matches.iterrows():
    home_team = match["HomeTeam"]
    away_team = match["AwayTeam"]
    home_goals = match["FTHG"]
    away_goals = match["FTAG"]

    if home_goals > away_goals:
        home_points, away_points, actual_home = 3, 0, 1.0
    elif home_goals < away_goals:
        home_points, away_points, actual_home = 0, 3, 0.0
    else:
        home_points, away_points, actual_home = 1, 1, 0.5

    expected_home = 1 / (
        1 + 10 ** ((ratings[away_team] - (ratings[home_team] + HOME_ADVANTAGE)) / 400)
    )

    ratings[home_team] += K_FACTOR * (actual_home - expected_home)
    ratings[away_team] += K_FACTOR * ((1 - actual_home) - (1 - expected_home))

    team_history[home_team].append({
        "goals_for": home_goals,
        "goals_against": away_goals,
        "points": home_points,
    })
    team_history[away_team].append({
        "goals_for": away_goals,
        "goals_against": home_goals,
        "points": away_points,
    })

def recent_stats(team, last_n=5):
    recent = team_history[team][-last_n:]

    return {
        "form_points_5": sum(game["points"] for game in recent),
        "avg_goals_for_5": np.mean([game["goals_for"] for game in recent]),
        "avg_goals_against_5": np.mean([game["goals_against"] for game in recent]),
    }

def make_prediction(home_team, away_team):
    home = recent_stats(home_team)
    away = recent_stats(away_team)

    input_data = pd.DataFrame([{
        "home_form_points_5": home["form_points_5"],
        "away_form_points_5": away["form_points_5"],
        "home_avg_goals_for_5": home["avg_goals_for_5"],
        "away_avg_goals_for_5": away["avg_goals_for_5"],
        "home_avg_goals_against_5": home["avg_goals_against_5"],
        "away_avg_goals_against_5": away["avg_goals_against_5"],
        "elo_difference": ratings[home_team] - ratings[away_team],
    }])

    X = sm.add_constant(input_data[feature_columns], has_constant="add")

    home_xg = home_model.predict(X).iloc[0]
    away_xg = away_model.predict(X).iloc[0]

    score_rows = []

    for home_goals in range(8):
        for away_goals in range(8):
            probability = (
                poisson.pmf(home_goals, home_xg) *
                poisson.pmf(away_goals, away_xg)
            )

            score_rows.append({
                "Score": f"{home_goals}-{away_goals}",
                "Probability": probability,
                "HomeGoals": home_goals,
                "AwayGoals": away_goals,
            })

    scores = pd.DataFrame(score_rows)

    home_win = scores[scores["HomeGoals"] > scores["AwayGoals"]]["Probability"].sum()
    draw = scores[scores["HomeGoals"] == scores["AwayGoals"]]["Probability"].sum()
    away_win = scores[scores["HomeGoals"] < scores["AwayGoals"]]["Probability"].sum()

    top_scores = scores.sort_values("Probability", ascending=False).head(5).copy()
    top_scores["Probability"] = (top_scores["Probability"] * 100).round(1)

    return home_xg, away_xg, home_win, draw, away_win, top_scores

teams = sorted(matches["HomeTeam"].unique())

home_team = st.selectbox("Home team", teams, index=teams.index("Arsenal"))
away_options = [team for team in teams if team != home_team]
away_team = st.selectbox(
    "Away team",
    away_options,
    index=away_options.index("Chelsea") if "Chelsea" in away_options else 0,
)

if st.button("Predict match", type="primary", use_container_width=True):
    home_xg, away_xg, home_win, draw, away_win, top_scores = make_prediction(
        home_team, away_team
    )

    st.subheader(f"{home_team} vs {away_team}")
    st.write(f"Expected goals: **{home_team} {home_xg:.2f} — {away_team} {away_xg:.2f}**")

    first, second, third = st.columns(3)
    first.metric("Home win", f"{home_win:.1%}")
    second.metric("Draw", f"{draw:.1%}")
    third.metric("Away win", f"{away_win:.1%}")

    st.subheader("Most likely exact scores")
    st.dataframe(
        top_scores[["Score", "Probability"]],
        hide_index=True,
        use_container_width=True,
    )

st.divider()
st.caption(
    f"Data current through {matches['Date'].max().date()}. "
    "Predictions are probabilities, not guarantees."
)



Writing app.py


In [3]:
from importlib.metadata import version
from pathlib import Path

packages = ["streamlit", "pandas", "numpy", "scipy", "statsmodels"]

requirements = "\n".join(
    f"{package}=={version(package)}"
    for package in packages
)

Path("requirements.txt").write_text(requirements + "\n")

print(requirements)

streamlit==1.64.0
pandas==2.2.2
numpy==2.0.0
scipy==1.18.1
statsmodels==0.15.0


In [5]:
%%writefile .gitignore
__pycache__/
.ipynb_checkpoints/
*.pyc
data/raw/
data/processed/epl_features.csv
data/processed/epl_enhanced_features.csv
models/xgb_home_goals.joblib
models/xgb_away_goals.joblib
outputs/

Writing .gitignore


In [4]:
%%writefile README.md
# Premier League Score Predictor

A machine-learning web app that estimates Premier League match scores, expected goals, and home-win/draw/away-win probabilities.

## Model

The app uses a Poisson regression model with:

- Each team's form from its previous five matches
- Recent goals scored and conceded
- Elo-based team-strength difference
- Home advantage

The model produces expected goals, exact-score probabilities, and match-result probabilities.

## Performance

On an unseen Premier League test season, the Elo-enhanced model achieved:

- Home-goal MAE: 0.970
- Away-goal MAE: 0.886
- Exact-score accuracy: 10.7%
- Match-result accuracy: 51.2%

## Data

The model uses completed Premier League results through 20 September 2026.

## Run locally

```bash
python -m pip install -r requirements.txt
python -m streamlit run app.py

Writing README.md
